In [8]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from Utils.TrainUtils import get_device

from Utils.DataUtils import FlatTransactionDataset, PREPROCESSED_NORM_DIRECTORY_PATH

from Models.AutoEncoder import AutoEncoder, GatedAutoEncoder

Just doing a simplified training loop as it seems like a lot of work to make dedicated train functions just for this

In [9]:
device = get_device()
print(device)
batch_size = 64
shuffle_train = True
num_workers = 0
pin_memory = True

non_fraud_dataset_train = FlatTransactionDataset(processed_dir = PREPROCESSED_NORM_DIRECTORY_PATH,
                                                 split="train",
                                                 filter_mode="nonfraud_only",
                                                 return_labels=False
                                                )

non_fraud_dataset_val = FlatTransactionDataset(processed_dir = PREPROCESSED_NORM_DIRECTORY_PATH,
                                                 split="val",
                                                 filter_mode="nonfraud_only",
                                                 return_labels=False
                                                )

all_fraud_dataset_test = FlatTransactionDataset(processed_dir = PREPROCESSED_NORM_DIRECTORY_PATH,
                                                 split="test",
                                                 filter_mode="all",
                                                 return_labels=True
                                                )

feature_dim = non_fraud_dataset_train.feature_dim

train_loader = DataLoader(
        non_fraud_dataset_train,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

val_loader = DataLoader(
        non_fraud_dataset_val,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

test_loader = DataLoader(
        all_fraud_dataset_test,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

cuda


In [10]:
feature_names = non_fraud_dataset_train.feature_keys

condition_no_pca_c3 = lambda name: name.startswith("v_pca_") or (name == "C3")
condition_no_pca = lambda name: name.startswith("v_pca_")

removed_features = [name for name in feature_names if condition_no_pca(name)]

removed_set = set(removed_features)

keep_mask_np = np.array(
    [name not in removed_set for name in feature_names],
    dtype=bool
)

print("Original feature dim:", len(feature_names))
print("Filtered feature dim:", keep_mask_np.sum())
print(f"Removing {len(removed_features)} features")
print("Removed featuers:", removed_features)

Original feature dim: 111
Filtered feature dim: 71
Removing 40 features
Removed featuers: ['v_pca_1', 'v_pca_10', 'v_pca_11', 'v_pca_12', 'v_pca_13', 'v_pca_14', 'v_pca_15', 'v_pca_16', 'v_pca_17', 'v_pca_18', 'v_pca_19', 'v_pca_2', 'v_pca_20', 'v_pca_21', 'v_pca_22', 'v_pca_23', 'v_pca_24', 'v_pca_25', 'v_pca_26', 'v_pca_27', 'v_pca_28', 'v_pca_29', 'v_pca_3', 'v_pca_30', 'v_pca_31', 'v_pca_32', 'v_pca_33', 'v_pca_34', 'v_pca_35', 'v_pca_36', 'v_pca_37', 'v_pca_38', 'v_pca_39', 'v_pca_4', 'v_pca_40', 'v_pca_5', 'v_pca_6', 'v_pca_7', 'v_pca_8', 'v_pca_9']


In [11]:
keep_mask_t = torch.tensor(keep_mask_np, dtype=torch.bool, device=device)
filtered_feature_dim = int(keep_mask_np.sum())

In [12]:
def compute_feature_mse(model, loader, keep_mask_t, kept_feature_names, device, gated = False):
    model.eval()

    feature_dim = len(kept_feature_names)
    feature_sse = torch.zeros(feature_dim, device=device)
    num_samples = 0

    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device, non_blocking=True)
            x = x[:, keep_mask_t]

            if gated:
                x_hat, _, _ = model(x)
            else:
                x_hat, _ = model(x)

            sq_err = (x_hat - x) ** 2
            feature_sse += sq_err.sum(dim=0)
            num_samples += x.size(0)

    feature_mse = (feature_sse / num_samples).detach().cpu().numpy()
    return feature_mse

In [13]:
# model_AE = AutoEncoder(input_dim=feature_dim,
#                        noise_std=0.02
#                        ).to(device)

model_AE = AutoEncoder(input_dim=filtered_feature_dim,
                       noise_std=0.02
                       ).to(device)

optimizer = torch.optim.Adam(model_AE.parameters(), lr=1e-3)
num_epochs = 15

for epoch in range(num_epochs):
    model_AE.train()
    total_loss = 0.0

    for batch in train_loader:
        x = batch["x"].to(device, non_blocking=True)
        x = x[:, keep_mask_t]   # remove all v_pca_* columns

        optimizer.zero_grad()

        x_hat, _ = model_AE(x)
        loss = F.mse_loss(x_hat, x)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model_AE.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            x = batch["x"].to(device, non_blocking=True)
            x = x[:, keep_mask_t]   # same filtering for validation

            x_hat, _ = model_AE(x)
            loss = F.mse_loss(x_hat, x)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} - "
        f"train_loss: {avg_train_loss:.6f} - "
        f"val_loss: {avg_val_loss:.6f}"
    )

    kept_feature_names = [
        name for name, keep in zip(feature_names, keep_mask_np)
        if keep
    ]

    train_feature_mse = compute_feature_mse(
        model_AE, train_loader, keep_mask_t, kept_feature_names, device
    )
    val_feature_mse = compute_feature_mse(
        model_AE, val_loader, keep_mask_t, kept_feature_names, device
    )

    rows = []
    for name, train_mse, val_mse in zip(kept_feature_names, train_feature_mse, val_feature_mse):
        rows.append((name, train_mse, val_mse, val_mse - train_mse))

    rows.sort(key=lambda x: x[2], reverse=True)

    print("Top 20 kept features by validation reconstruction MSE:")
    for name, train_mse, val_mse, gap in rows[:20]:
        print(
            f"{name:30s} "
            f"train={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"gap={val_mse - train_mse:.6f}"
        )



Epoch 1/15 - train_loss: 0.162534 - val_loss: 0.035313
Top 20 kept features by validation reconstruction MSE:
card6                          train=0.151071 val=0.149472 gap=-0.001599
C3                             train=0.047182 val=0.125976 gap=0.078794
R_emaildomain                  train=0.094434 val=0.088371 gap=-0.006063
M5                             train=0.084392 val=0.087829 gap=0.003437
D1                             train=0.060362 val=0.085669 gap=0.025307
D13                            train=0.038027 val=0.084900 gap=0.046873
addr1                          train=0.055503 val=0.079140 gap=0.023636
card4                          train=0.060819 val=0.077000 gap=0.016181
D3                             train=0.073180 val=0.069959 gap=-0.003221
M2                             train=0.046657 val=0.060185 gap=0.013528
P_emaildomain                  train=0.055594 val=0.060007 gap=0.004414
M4                             train=0.045761 val=0.051909 gap=0.006149
C5                     

In [14]:
model_GAE = GatedAutoEncoder(input_dim=filtered_feature_dim,
                             noise_std=0.02).to(device)

optimizer = torch.optim.Adam(model_GAE.parameters(), lr=1e-3)
num_epochs = 15

for epoch in range(num_epochs):
    model_GAE.train()
    total_loss = 0.0

    for batch in train_loader:
        x = batch["x"].to(device, non_blocking=True)
        x = x[:, keep_mask_t]   # remove all v_pca_* columns

        optimizer.zero_grad()

        x_hat, _, _ = model_GAE(x)
        loss = F.mse_loss(x_hat, x)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model_GAE.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            x = batch["x"].to(device, non_blocking=True)
            x = x[:, keep_mask_t]   # same filtering for validation

            x_hat, _, _ = model_GAE(x)
            loss = F.mse_loss(x_hat, x)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} - "
        f"train_loss: {avg_train_loss:.6f} - "
        f"val_loss: {avg_val_loss:.6f}"
    )

    kept_feature_names = [
        name for name, keep in zip(feature_names, keep_mask_np)
        if keep
    ]

    train_feature_mse = compute_feature_mse(
        model_GAE, train_loader, keep_mask_t, kept_feature_names, device, gated=True
    )
    val_feature_mse = compute_feature_mse(
        model_GAE, val_loader, keep_mask_t, kept_feature_names, device, gated=True
    )

    rows = []
    for name, train_mse, val_mse in zip(kept_feature_names, train_feature_mse, val_feature_mse):
        rows.append((name, train_mse, val_mse, val_mse - train_mse))

    rows.sort(key=lambda x: x[2], reverse=True)

    print("Top 20 kept features by validation reconstruction MSE:")
    for name, train_mse, val_mse, gap in rows[:20]:
        print(
            f"{name:30s} "
            f"train={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"gap={val_mse - train_mse:.6f}"
        )

Epoch 1/15 - train_loss: 0.190968 - val_loss: 0.099903
Top 20 kept features by validation reconstruction MSE:
C3                             train=0.879486 val=3.344894 gap=2.465408
R_emaildomain                  train=0.266441 val=0.706430 gap=0.439989
D8                             train=0.097686 val=0.269638 gap=0.171952
P_emaildomain                  train=0.162552 val=0.265486 gap=0.102935
card6                          train=0.159714 val=0.163265 gap=0.003550
TransactionAmt                 train=0.086266 val=0.104186 gap=0.017920
M5                             train=0.086989 val=0.087980 gap=0.000992
M3                             train=0.062047 val=0.070847 gap=0.008800
D10                            train=0.046328 val=0.068788 gap=0.022460
card4                          train=0.048689 val=0.068486 gap=0.019797
D13                            train=0.045915 val=0.065900 gap=0.019985
addr1                          train=0.037162 val=0.060219 gap=0.023057
C9                        

testing the latent representation of our auto-encoder

In [15]:
from sklearn.metrics import roc_auc_score, average_precision_score

def describe(name, x):
    print(f"\n{name}")
    print("mean:", x.mean().item())
    print("median:", x.median().item())
    print("90th pct:", x.quantile(0.9).item())
    print("95th pct:", x.quantile(0.95).item())

In [20]:
model_AE.eval()
model_GAE.eval()

all_errors = []
all_errors_gated = []
all_labels = []


with torch.no_grad():
    for batch in test_loader:   
        x = batch["x"].to(device, non_blocking=True)
        y = batch["y"]          # keep on CPU for now

        x = x[:, keep_mask_t]   # apply the same filtering

        x_hat, _ = model_AE(x)
        x_hat_gated, _, _ = model_GAE(x)

        # per-sample reconstruction error
        err = ((x_hat - x) ** 2).mean(dim=1)   # [B]
        err_gated = ((x_hat_gated - x) ** 2).mean(dim=1)   # [B]

        # err = torch.clamp(err, max=1.0) # some normal transactions are completely messed up
        err = torch.log1p(err) # trying other clipping methods for extreme outliers
        err_gated = torch.log1p(err_gated) # trying other clipping methods for extreme outliers

        all_errors_gated.append(err_gated.cpu())
        all_errors.append(err.cpu())
        all_labels.append(y)

# concatenate
all_errors_gated = torch.cat(all_errors_gated)
all_errors = torch.cat(all_errors)
all_labels = torch.cat(all_labels)

gated_normal_errors = all_errors_gated[all_labels == 0]
gated_fraud_errors = all_errors_gated[all_labels == 1]

normal_errors = all_errors[all_labels == 0]
fraud_errors = all_errors[all_labels == 1]

print("GatedAutoEncoder "+"="*10)
# Hopefully fraud mean error > Normal mean error
print("Normal mean error:", gated_normal_errors.mean().item())
print("Fraud mean error :", gated_fraud_errors.mean().item())
print("")
print("Normal std:", gated_normal_errors.std().item())
print("Fraud std :", gated_fraud_errors.std().item())
print("")
print("Normal max:", gated_normal_errors.max().item())
print("Fraud max :", gated_fraud_errors.max().item())
print("")
print("Normal 99th pct:", gated_normal_errors.quantile(0.99).item())
print("Fraud 99th pct :", gated_fraud_errors.quantile(0.99).item())

# Print out percentiles for distribution
print("")
describe("Normal", gated_normal_errors)
describe("Fraud", gated_fraud_errors)

gated_roc_auc = roc_auc_score(all_labels.numpy(), all_errors_gated.numpy())
gated_pr_auc = average_precision_score(all_labels.numpy(), all_errors_gated.numpy())

print("")
print("ROC AUC:", gated_roc_auc)
print("PR AUC:", gated_pr_auc)

threshold_gated = all_errors_gated.quantile(0.95)

high_error_gated = all_errors_gated >= threshold_gated
fraud_rate_high_gated = all_labels[high_error_gated].float().mean()

print("")
print("Fraud rate in top 5% error:", fraud_rate_high_gated.item())

print("\nAutoEncoder "+"="*10)
# Hopefully fraud mean error > Normal mean error
print("Normal mean error:", normal_errors.mean().item())
print("Fraud mean error :", fraud_errors.mean().item())
print("")
print("Normal std:", normal_errors.std().item())
print("Fraud std :", fraud_errors.std().item())
print("")
print("Normal max:", normal_errors.max().item())
print("Fraud max :", fraud_errors.max().item())
print("")
print("Normal 99th pct:", normal_errors.quantile(0.99).item())
print("Fraud 99th pct :", fraud_errors.quantile(0.99).item())

# Print out percentiles for distribution
print("")
describe("Normal", normal_errors)
describe("Fraud", fraud_errors)

roc_auc = roc_auc_score(all_labels.numpy(), all_errors.numpy())
pr_auc = average_precision_score(all_labels.numpy(), all_errors.numpy())

print("")
print("ROC AUC:", roc_auc)
print("PR AUC:", pr_auc)

threshold = all_errors.quantile(0.95)

high_error = all_errors >= threshold
fraud_rate_high = all_labels[high_error].float().mean()

print("")
print("Fraud rate in top 5% error:", fraud_rate_high.item())

GatedAutoEncoder ==========
Normal mean error: 0.008522900752723217
Fraud mean error : 0.015389794483780861

Normal std: 0.05079435557126999
Fraud std : 0.04284244775772095

Normal max: 5.409883975982666
Fraud max : 1.034340500831604

Normal 99th pct: 0.0789647102355957
Fraud 99th pct : 0.18415583670139313


Normal
mean: 0.008522900752723217
median: 0.0033692377619445324
90th pct: 0.014605913311243057
95th pct: 0.024965593591332436

Fraud
mean: 0.015389794483780861
median: 0.00513148307800293
90th pct: 0.027247905731201172
95th pct: 0.05292074754834175

ROC AUC: 0.6099623242803188
PR AUC: 0.059584867082981635

Fraud rate in top 5% error: 0.0796189159154892

AutoEncoder ==========
Normal mean error: 0.005944633856415749
Fraud mean error : 0.011581039987504482

Normal std: 0.052338406443595886
Fraud std : 0.034740883857011795

Normal max: 6.103795051574707
Fraud max : 0.6132092475891113

Normal 99th pct: 0.04539547860622406
Fraud 99th pct : 0.19647347927093506


Normal
mean: 0.0059446338

In [31]:
for p in range(0, 100, 5):
    q = p/100
    thresh = all_errors.quantile(q)
    thresh_gated = all_errors_gated.quantile(q)
    rate = all_labels[all_errors >= thresh].float().mean()
    rate_gated = all_labels[all_errors_gated >= thresh].float().mean()
    print(f"(AutoEncoder) Top {int((1-q)*100)}% fraud rate: {rate.item()} vs (GatedAutoEncoder) Top {int((1-q)*100)}% fraud rate: {rate_gated.item()}")

(AutoEncoder) Top 100% fraud rate: 0.03717314079403877 vs (GatedAutoEncoder) Top 100% fraud rate: 0.03717314079403877
(AutoEncoder) Top 95% fraud rate: 0.038252148777246475 vs (GatedAutoEncoder) Top 95% fraud rate: 0.03748467564582825
(AutoEncoder) Top 90% fraud rate: 0.039054080843925476 vs (GatedAutoEncoder) Top 90% fraud rate: 0.037913598120212555
(AutoEncoder) Top 85% fraud rate: 0.04011048376560211 vs (GatedAutoEncoder) Top 85% fraud rate: 0.038360320031642914
(AutoEncoder) Top 80% fraud rate: 0.04108627513051033 vs (GatedAutoEncoder) Top 80% fraud rate: 0.03879960626363754
(AutoEncoder) Top 75% fraud rate: 0.042328283190727234 vs (GatedAutoEncoder) Top 75% fraud rate: 0.039161112159490585
(AutoEncoder) Top 70% fraud rate: 0.043310243636369705 vs (GatedAutoEncoder) Top 70% fraud rate: 0.0399942584335804
(AutoEncoder) Top 65% fraud rate: 0.044469453394412994 vs (GatedAutoEncoder) Top 65% fraud rate: 0.04098573327064514
(AutoEncoder) Top 60% fraud rate: 0.045453257858753204 vs (Gate

In [30]:
torch.save({
    "model_state_dict": model_AE.state_dict(),
    "input_dim": filtered_feature_dim,
    "latent_dim": model_AE.latent_dim,
    "hidden_dims": [256, 128],
    "noise_std": model_AE.noise_std,
    "mask": keep_mask_np,
    "feature_names": feature_names,
    "kept_feature_names": kept_feature_names,
    "model_type": "AutoEncoder",
}, "checkpoints/ae_clean.pt")

torch.save({
    "model_state_dict": model_GAE.state_dict(),
    "input_dim": filtered_feature_dim,
    "latent_dim": model_GAE.latent_dim,
    "hidden_dims": [256, 128],
    "noise_std": model_GAE.noise_std,
    "mask": keep_mask_np,
    "feature_names": feature_names,
    "kept_feature_names": kept_feature_names,
    "model_type": "GatedAutoEncoder",
}, "checkpoints/gae_clean.pt")